# Customer Shopping Behaviour Analysis

In [1]:
import pandas as pd

In [7]:
customers = pd.read_csv("customer_shopping_behavior.csv")
print("File Added successfully")

File Added successfully


In [9]:
customers.head(5)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [11]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [13]:
customers.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [15]:
customers.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

## Handling Missing Review Ratings

The **`review_rating`** column contains **37 missing values**.

A common approach for handling missing numerical values is to impute them using the **mean** or **median**.

- **Mean imputation** is generally suitable when the data is normally distributed. However, review ratings can be influenced by outliers, making the mean less representative of the typical rating.
- **Median imputation** is more robust because it is not affected by extreme values and better preserves the original rating distribution.

However, using a **single overall median** for all missing values is not the best approach. Different product categories receive ratings based on different customer expectations. For example:

- **Clothing** is often rated based on factors such as design, fit, and appearance.
- **Footwear** is commonly rated based on comfort, durability, and quality.

Since customer rating behavior varies across categories, imputing all missing values with one overall median could introduce bias.

**Approach Used:**
- Calculate the **median review rating for each product category**.
- Replace missing ratings with the **median of the corresponding category**.

This approach preserves category-specific rating patterns and provides a more realistic estimate for the missing values.

In [21]:
customers['Review Rating'] = customers.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [23]:
customers.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

## Standardizing Column Names

Some column names contained spaces (e.g., `customer ID`), which is not a best practice, especially when working with SQL.

**Approach Used:**
- Converted all column names to **lowercase**.
- Replaced spaces with **underscores (`_`)** (e.g., `customer ID` → `customer_id`).

This improves consistency, readability, and SQL compatibility.

In [34]:
customers.columns = customers.columns.str.lower()
customers.columns = customers.columns.str.replace(' ','_')
customers = customers.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [36]:
customers.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

# Featured Engineering

## Create a column age_group

In [44]:
labels = ['Young Adult','Adult','Middle-aged','Senior']
customers['age_group'] = pd.qcut(customers['age'],q=4,labels = labels)

In [46]:
customers[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


## Create column purchase_frequency_days

In [50]:
frequency_mapping = {
    'Fortnightly' : 14,
    'Weekly' : 7,
    'Monthly' : 30,
    'Quarterly' : 90,
    'Bi-Weekly' : 14,
    'Annually' : 365,
    'Every 3 Months' : 90
}

customers['purchase_frequency_days'] = customers['frequency_of_purchases'].map(frequency_mapping)

In [54]:
customers[['purchase_frequency_days','frequency_of_purchases']].head(5)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually


In [56]:
customers[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


## Comparing `promo_code` and `discount`

These two columns appear related, but they may not always contain the same information.

In most cases, a **promo code** is used to apply a discount. However, companies also offer **direct discounts** during events such as festivals, sales, or special promotions without requiring a promo code.

**Approach Used:**
- Compared the `promo_code` and `discount` columns to check whether they always correspond or if discounts were applied without a promo code.

In [63]:
(customers['discount_applied']==customers['promo_code_used']).all()

np.True_

## Dropping `promo_code`

The analysis showed that **`promo_code`** and **`discount`** contain identical information.

Since `promo_code` does not provide any additional value, it is a **redundant column** and can be safely removed to simplify the dataset.

In [68]:
customers = customers.drop('promo_code_used', axis =1)

In [72]:
customers.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [94]:
customers.to_csv("cleaned_customers.csv" , index = False)

In [98]:
customers.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


## Loading Data into SQL Server

After completing the data cleaning and preprocessing steps, the cleaned dataset is loaded into **SQL Server** for further analysis and querying.

In [84]:
!pip install pyodbc sqlalchemy

In [86]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

In [88]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=SOURABH-05MAR96\SQLEXPRESS;"
    "DATABASE=Customer_sales_data;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [92]:
customers.to_sql(
    name='customer',
    con=engine,
    if_exists='replace',
    index=False
)

50